# HYROX distributed scraper — one Colab node = one shard

Each Colab runtime has its **own IP**, so each gets its own rate-limit budget. Run several copies of this notebook in parallel (different tabs/accounts), one per shard.

**One-time setup (do this once, locally):**
1. Generate shards: `python hyrox_scraper.py --make-shards 8 --out ./data_v2`
2. In Google Drive create a folder `MyDrive/hyrox/` and upload into it:
   - `hyrox_scraper.py`
   - the whole `shards/` folder (shard_00.json … shard_07.json)

**Per node:** open this notebook, set `SHARD` below to a different number (0,1,2,…), and Run all.
It is **resumable** — if Colab disconnects, just re-run; finished events are skipped.

In [ ]:
#@title 1. Install dependencies
!pip -q install requests beautifulsoup4 lxml pandas pyarrow playwright
!python -m playwright install chromium
!python -m playwright install-deps chromium 2>/dev/null || true

In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 3. Pick this node's shard
SHARD = 0  #@param {type:"integer"}   <-- CHANGE per node: 0,1,2,...
import os, shutil
BASE = '/content/drive/MyDrive/hyrox'
shard_file = f'{BASE}/shards/shard_{SHARD:02d}.json'
out_dir    = f'{BASE}/node_{SHARD:02d}'
assert os.path.exists(f'{BASE}/hyrox_scraper.py'), 'Upload hyrox_scraper.py to MyDrive/hyrox/'
assert os.path.exists(shard_file), f'Missing {shard_file} — upload the shards/ folder'
shutil.copy(f'{BASE}/hyrox_scraper.py', '/content/hyrox_scraper.py')
import json; print('events in this shard:', len(json.load(open(shard_file))))
print('writing to:', out_dir)

In [ ]:
#@title 4. Scrape this shard (gentle: 1 session, ~2.5 req/s)
# ~2.5/s is sustainable on one IP. `python -u` = live (unbuffered) logs.
# Big events now log progress every 100 athletes, so you can see it working.
# If a node still gets blocked it auto-pauses ~2.5 min then resumes.
# Re-run this cell any time to continue (it skips finished events).
!cd /content && python -u hyrox_scraper.py --events-file "{shard_file}" --out "{out_dir}" --sessions 1 --rate 2.5 --workers 4

## When ALL nodes are done

Run the cell below on **any one** node (or locally). `--combine` recursively picks up every `node_*/by_event/` folder in Drive, de-dupes, and writes the final `hyrox_full.csv/.parquet/.jsonl` + `DATA_DICTIONARY.md` into `MyDrive/hyrox/`.

In [ ]:
#@title 5. (run once, at the end) Merge all shards into the final dataset
!cd /content && python hyrox_scraper.py --combine --out "/content/drive/MyDrive/hyrox"